# DL_S01 — Why Deep Learning Changed Everything

## Opening hook

**Decision problem:** why can a model understand images, text, and signals better than hand-made rules?

Classic machine learning often starts with this question:

> What features should we design?

Deep learning changed the question:

> What representation can the model learn from data?

This notebook is not a deep learning specialization. It is a bridge: from classic ML to modern AI systems.

## Teaching angle

Not: memorize neural-network formulas.

Instead:

> When manual features stop being enough, learned representations become the product.

We use the built-in scikit-learn digits dataset: small images of handwritten digits, no download, no GPU, no heavy dependency.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "notebooks").exists():
        ROOT = candidate
        break

OUT = ROOT / "outputs" / "modern_ai_deep_learning"
OUT.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT)

# 1. Classic ML reminder

A classic ML workflow often depends on manually designed features.

For images, a human might invent features such as:

- total ink intensity;
- ink in each quadrant;
- left-right symmetry;
- top-bottom symmetry;
- center intensity.

These features are understandable. They are also limited.

In [ ]:
digits = load_digits()
X_images = digits.images
X_pixels = digits.data / 16.0
y = digits.target

print("Images:", X_images.shape)
print("Flat pixels:", X_pixels.shape)

fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for ax, image, label in zip(axes.ravel(), X_images[:12], y[:12]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
fig.suptitle("Tiny handwritten digit images")
fig.tight_layout()
plt.show()

In [ ]:
def handcrafted_features(images):
    rows = []
    for img in images:
        top = img[:4, :]
        bottom = img[4:, :]
        left = img[:, :4]
        right = img[:, 4:]
        center = img[2:6, 2:6]
        rows.append({
            "total_ink": img.sum(),
            "top_ink": top.sum(),
            "bottom_ink": bottom.sum(),
            "left_ink": left.sum(),
            "right_ink": right.sum(),
            "center_ink": center.sum(),
            "vertical_balance": top.sum() - bottom.sum(),
            "horizontal_balance": left.sum() - right.sum(),
            "nonzero_pixels": (img > 0).sum(),
        })
    return pd.DataFrame(rows)

X_manual = handcrafted_features(X_images)
X_manual.head()

In [ ]:
X_train_manual, X_test_manual, y_train, y_test = train_test_split(
    X_manual, y, test_size=0.25, random_state=42, stratify=y
)

classic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
classic_model.fit(X_train_manual, y_train)
classic_train_pred = classic_model.predict(X_train_manual)
classic_test_pred = classic_model.predict(X_test_manual)

classic_metrics = {
    "model": "classic_ml_handcrafted_features",
    "train_accuracy": accuracy_score(y_train, classic_train_pred),
    "test_accuracy": accuracy_score(y_test, classic_test_pred),
    "representation": "human-designed summary features",
}
classic_metrics

## Limits of hand-crafted representations

The manual features are interpretable, but they throw away shape.

A `3`, `5`, and `8` can have similar ink totals. The difference is not just how much ink exists. It is where strokes connect, curve, and leave empty space.

That structure is hard to fully design by hand.

# 2. Representation learning intuition

Deep learning learns intermediate representations.

A simple mental model:

- **input:** raw pixels, words, sounds, signals;
- **layers:** transformations learned from data;
- **early features:** simple patterns;
- **deeper features:** more useful abstractions;
- **embedding / representation:** compressed meaning useful for a task.

This matters because modern AI products are often representation products:

- text search uses semantic embeddings;
- vision systems learn visual features;
- recommenders learn user/item representations;
- transformers learn contextual representations;
- LLMs scale representation learning across language and tools.

# 3. Minimal neural network demo

We use `MLPClassifier`, a lightweight neural network included in scikit-learn.

This is not industrial deep learning. It is a runnable demonstration of the key idea:

> Let the model learn useful transformations from raw-ish inputs instead of forcing humans to define every feature.

In [ ]:
X_train_pixels, X_test_pixels, y_train_pixels, y_test_pixels = train_test_split(
    X_pixels, y, test_size=0.25, random_state=42, stratify=y
)

neural_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(hidden_layer_sizes=(64,), activation="relu", max_iter=300, random_state=42, early_stopping=True, validation_fraction=0.15)),
])
neural_model.fit(X_train_pixels, y_train_pixels)
neural_train_pred = neural_model.predict(X_train_pixels)
neural_test_pred = neural_model.predict(X_test_pixels)

neural_metrics = {
    "model": "lightweight_neural_network_raw_pixels",
    "train_accuracy": accuracy_score(y_train_pixels, neural_train_pred),
    "test_accuracy": accuracy_score(y_test_pixels, neural_test_pred),
    "representation": "learned hidden representation from pixels",
}
neural_metrics

In [ ]:
metrics = pd.DataFrame([classic_metrics, neural_metrics])
metrics[["train_accuracy", "test_accuracy"]] = metrics[["train_accuracy", "test_accuracy"]].round(3)
metrics.to_csv(OUT / "model_metrics.csv", index=False)
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metrics.plot.bar(x="model", y=["train_accuracy", "test_accuracy"], ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("Manual features vs learned representation")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", labelrotation=20)
fig.tight_layout()
fig.savefig(OUT / "manual_vs_learned_representation.png", dpi=140)
plt.show()

# 4. Overfitting and trust

Deep learning is not magic.

A bigger model can memorize. A model that looks strong on training data can fail on validation data.

Trust comes from evaluation, not architecture hype.

In [ ]:
overfit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(hidden_layer_sizes=(128, 128), activation="relu", max_iter=500, random_state=7, early_stopping=False)),
])
overfit_model.fit(X_train_pixels, y_train_pixels)
overfit_train = accuracy_score(y_train_pixels, overfit_model.predict(X_train_pixels))
overfit_test = accuracy_score(y_test_pixels, overfit_model.predict(X_test_pixels))

overfit_check = pd.DataFrame([
    {"model": "regularized_mlp_early_stopping", "train_accuracy": neural_metrics["train_accuracy"], "test_accuracy": neural_metrics["test_accuracy"], "generalization_gap": neural_metrics["train_accuracy"] - neural_metrics["test_accuracy"]},
    {"model": "larger_mlp_no_early_stopping", "train_accuracy": overfit_train, "test_accuracy": overfit_test, "generalization_gap": overfit_train - overfit_test},
]).round(3)
overfit_check.to_csv(OUT / "overfitting_check.csv", index=False)
overfit_check

In [ ]:
print(classification_report(y_test_pixels, neural_test_pred, zero_division=0))
cm = confusion_matrix(y_test_pixels, neural_test_pred)
cm_df = pd.DataFrame(cm, index=[f"actual_{i}" for i in range(10)], columns=[f"pred_{i}" for i in range(10)])
cm_df.head()

# 5. Modern AI bridge

Deep learning changed modern AI because representation learning scales.

The same core idea appears across systems:

- **NLP:** words become embeddings, then contextual representations;
- **computer vision:** pixels become edges, shapes, objects, scenes;
- **recommendations:** users and items become vectors in a shared space;
- **transformers:** attention learns relationships across tokens, patches, or modalities;
- **multimodal systems:** text, images, audio, and actions can be mapped into interoperable representations;
- **LLMs:** representation learning at massive scale, compressed into reusable models.

The product shift is profound: the model is no longer just a predictor. It becomes a reusable representation engine.

In [ ]:
representation_note = """# Representation Learning Note

Classic ML uses features humans design: totals, ratios, flags, bins, aggregations.

Deep learning learns intermediate representations from data. For images, it can learn stroke patterns; for text, semantic relationships; for recommendations, user/item similarity.

This is why deep learning changed modern AI: when manual features stop being enough, learned representations become the product.
"""
(OUT / "representation_learning_note.md").write_text(representation_note)

modern_ai_bridge = """# Modern AI Bridge Summary

Deep learning is not magic. It is representation learning at scale.

The bridge from classic ML to modern AI is the move from hand-crafted features to learned features:
- classic ML: humans design features, model learns decision boundary;
- deep learning: model learns useful representations and decision boundary;
- transformers/LLMs: representation learning scales across language, images, tools, and context.

Trust still requires validation, error analysis, and product judgment.
"""
(OUT / "modern_ai_bridge_summary.md").write_text(modern_ai_bridge)

print(representation_note)
print(modern_ai_bridge)

# Final lesson

Deep learning is not magic.

It is representation learning at scale.

Students should remember:

> **Classic ML uses features we design. Deep learning learns features we could not easily design.**